# Customer Gain vs Loss (2025)
Counts distinct customers acquired vs churned using `fact_opportunity`.
- **Gained**: `close_status = 'Won'` with contract `start_date` in 2025.
- **Lost**: `close_status = 'Lost'` with contract `end_date` in 2025.

In [ ]:
%%sql
WITH gained AS (
    SELECT COUNT(DISTINCT customer_id) AS customers_gained
    FROM fact_opportunity
    WHERE close_status = 'Won'
      AND start_date >= DATE '2025-01-01'
      AND start_date < DATE '2026-01-01'
), lost AS (
    SELECT COUNT(DISTINCT customer_id) AS customers_lost
    FROM fact_opportunity
    WHERE close_status = 'Lost'
      AND end_date >= DATE '2025-01-01'
      AND end_date < DATE '2026-01-01'
)
SELECT
    g.customers_gained,
    l.customers_lost
FROM gained g
CROSS JOIN lost l;

## Curated Revenue Base
This view joins `fact_opportunity` with customer, product, employee, and FX dimensions to expose the fields needed for every KPI (net logo growth, churn, expansion/contraction, ARR trends, and rolling revenue). It stores values in GBP so downstream KPI queries stay consistent.

In [ ]:

CREATE OR REPLACE VIEW curated_customer_revenue AS
WITH latest_fx AS (
    SELECT
        currency_code,
        fx_rate_to_gbp,
        effective_date,
        ROW_NUMBER() OVER (PARTITION BY currency_code ORDER BY effective_date DESC) AS rn
    FROM dim_fx_rate
), fact_enriched AS (
    SELECT
        fo.opportunity_id,
        fo.customer_id,
        dc.customer_name,
        dc.country,
        dc.industry_type,
        dc.account_created_date,
        CASE dc.country
            WHEN 'UK' THEN 'GBP'
            WHEN 'US' THEN 'USD'
            WHEN 'FR' THEN 'EUR'
            WHEN 'DE' THEN 'EUR'
            WHEN 'IN' THEN 'INR'
            ELSE 'GBP'
        END AS currency_code,
        fo.product_id,
        dp.product_name,
        dp.plan_name,
        dp.billing_cycle,
        fo.employee_id,
        de.employee_name,
        de.role AS employee_role,
        de.region AS employee_region,
        fo.start_date,
        fo.end_date,
        DATE_TRUNC('year', fo.start_date) AS contract_year_start,
        DATE_TRUNC('month', fo.start_date) AS contract_month,
        fo.contract_term,
        fo.close_status,
        fo.revenue_amount,
        fo.revenue_amount * COALESCE(lfx.fx_rate_to_gbp, 1) AS revenue_amount_gbp,
        CASE WHEN fo.close_status = 'Won' THEN 1 ELSE 0 END AS is_won,
        CASE WHEN fo.close_status = 'Lost' THEN 1 ELSE 0 END AS is_lost,
        CASE
            WHEN fo.close_status = 'Won'
             AND DATE_PART('year', fo.start_date) = DATE_PART('year', dc.account_created_date)
             THEN 1 ELSE 0
        END AS is_new_logo,
        CASE
            WHEN fo.close_status = 'Lost'
             AND fo.end_date BETWEEN DATE_TRUNC('year', CURRENT_DATE)
                                 AND DATEADD('year', 1, DATE_TRUNC('year', CURRENT_DATE)) - INTERVAL '1 day'
             THEN 1 ELSE 0
        END AS is_current_year_churn
    FROM fact_opportunity fo
    LEFT JOIN dim_customer dc ON dc.customer_id = fo.customer_id
    LEFT JOIN dim_product dp ON dp.product_id = fo.product_id
    LEFT JOIN dim_employee de ON de.employee_id = fo.employee_id
    LEFT JOIN (
        SELECT currency_code, fx_rate_to_gbp
        FROM latest_fx
        WHERE rn = 1
    ) lfx ON lfx.currency_code = CASE dc.country
            WHEN 'UK' THEN 'GBP'
            WHEN 'US' THEN 'USD'
            WHEN 'FR' THEN 'EUR'
            WHEN 'DE' THEN 'EUR'
            WHEN 'IN' THEN 'INR'
            ELSE 'GBP'
        END
), base AS (
    SELECT
        fe.*,
        LAG(fe.revenue_amount_gbp) OVER (PARTITION BY fe.customer_id ORDER BY fe.start_date) AS prev_revenue_gbp
    FROM fact_enriched fe
 )
SELECT
    b.*,
    (b.revenue_amount_gbp - b.prev_revenue_gbp) AS revenue_delta_gbp,
    CASE
        WHEN b.close_status = 'Won' AND b.revenue_amount_gbp > COALESCE(b.prev_revenue_gbp, 0) THEN 'Expansion'
        WHEN b.close_status = 'Won' AND b.revenue_amount_gbp < COALESCE(b.prev_revenue_gbp, 0) THEN 'Contraction'
        WHEN b.close_status = 'Lost' THEN 'Churn'
        ELSE 'Flat'
    END AS revenue_movement,
    SUM(b.revenue_amount_gbp) OVER (
        PARTITION BY b.customer_id
        ORDER BY b.start_date
        RANGE BETWEEN INTERVAL '365 day' PRECEDING AND CURRENT ROW
    ) AS rolling_12m_rr_gbp,
    SUM(b.revenue_amount_gbp) OVER (
        PARTITION BY DATE_TRUNC('year', b.start_date)
    ) AS fiscal_year_rr_gbp
FROM base b;

In [ ]:
%%sql
SELECT *
FROM curated_customer_revenue
ORDER BY start_date
LIMIT 25;

## KPI Queries
These SQL snippets run on the `curated_customer_revenue` view so you can reuse a consistent grain and currency across every metric. Adjust the date boundaries in each cell to target the exact reporting window you care about.

### KPI 1 — Customers Gained vs Lost (Current Period)
Counts distinct customers with wins vs churn inside a configurable reporting window so you can refresh "this period" at will.

In [ ]:
%%sql
WITH params AS (
    SELECT
        DATE '2025-01-01' AS period_start,
        DATE '2026-01-01' AS period_end
), gained AS (
    SELECT COUNT(DISTINCT customer_id) AS customers_gained
    FROM curated_customer_revenue, params
    WHERE close_status = 'Won'
      AND start_date >= params.period_start
      AND start_date < params.period_end
), lost AS (
    SELECT COUNT(DISTINCT customer_id) AS customers_lost
    FROM curated_customer_revenue, params
    WHERE close_status = 'Lost'
      AND end_date >= params.period_start
      AND end_date < params.period_end
), combined AS (
    SELECT
        g.customers_gained,
        l.customers_lost,
        g.customers_gained - l.customers_lost AS net_logo_change
    FROM gained g
    CROSS JOIN lost l
    CROSS JOIN params
 )
SELECT * FROM combined;

### KPI 2 — Churn Trend vs Prior Period
Compares distinct churned customers in the current window vs the same length immediately prior so you can answer whether churn is accelerating.

In [ ]:
%%sql
WITH params AS (
    SELECT
        DATE '2025-01-01' AS period_start,
        DATE '2026-01-01' AS period_end
), current_period AS (
    SELECT COUNT(DISTINCT customer_id) AS current_churned
    FROM curated_customer_revenue c, params p
    WHERE c.close_status = 'Lost'
      AND c.end_date >= p.period_start
      AND c.end_date < p.period_end
), prior_period AS (
    SELECT COUNT(DISTINCT customer_id) AS prior_churned
    FROM curated_customer_revenue c, params p
    WHERE c.close_status = 'Lost'
      AND c.end_date >= p.period_start - INTERVAL '1 year'
      AND c.end_date < p.period_end - INTERVAL '1 year'
 )
SELECT
    p.period_start,
    p.period_end,
    p.period_start - INTERVAL '1 year' AS prior_period_start,
    p.period_end - INTERVAL '1 year' AS prior_period_end,
    cur.current_churned,
    pr.prior_churned,
    cur.current_churned - pr.prior_churned AS churn_delta
FROM params p
CROSS JOIN current_period cur
CROSS JOIN prior_period pr;

### KPI 3 — Highest-Value Customers (Recurring Revenue)
Ranks customers by GBP-normalized recurring revenue so you can see who drives the largest book of business.

In [ ]:
%%sql
SELECT
    customer_id,
    COALESCE(customer_name, 'Unknown Customer') AS customer_name,
    SUM(revenue_amount_gbp) AS total_rr_gbp,
    COUNT(DISTINCT product_id) AS product_lines,
    COUNT(DISTINCT CASE WHEN close_status = 'Won' THEN product_id END) AS active_products_won,
    MIN(start_date) AS first_contract_start,
    MAX(end_date) AS latest_contract_end
FROM curated_customer_revenue
GROUP BY 1, 2
ORDER BY total_rr_gbp DESC
LIMIT 20;

### KPI 4 — Customers Expanding Usage
Identifies accounts whose most recent contract shows positive revenue delta, signaling expansion motions.

In [ ]:
%%sql
WITH ranked AS (
    SELECT
        c.*,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY start_date DESC, opportunity_id DESC) AS rn
    FROM curated_customer_revenue c
)
SELECT
    customer_id,
    COALESCE(customer_name, 'Unknown Customer') AS customer_name,
    start_date AS latest_start_date,
    prev_revenue_gbp,
    revenue_amount_gbp AS current_revenue_gbp,
    revenue_delta_gbp,
    revenue_movement,
    product_name,
    plan_name,
    employee_name AS owner_name
FROM ranked
WHERE rn = 1
  AND revenue_delta_gbp > 0
ORDER BY revenue_delta_gbp DESC
LIMIT 25;

### KPI 5 — Customers Contracting or Churning
Tracks the inverse of KPI 4 by surfacing customers whose latest motion is contraction or churn so CS can intervene.

In [ ]:
%%sql
WITH ranked AS (
    SELECT
        c.*,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY start_date DESC, opportunity_id DESC) AS rn
    FROM curated_customer_revenue c
)
SELECT
    customer_id,
    COALESCE(customer_name, 'Unknown Customer') AS customer_name,
    start_date AS latest_start_date,
    prev_revenue_gbp,
    revenue_amount_gbp AS current_revenue_gbp,
    revenue_delta_gbp,
    revenue_movement,
    close_status,
    product_name,
    plan_name,
    employee_name AS owner_name
FROM ranked
WHERE rn = 1
  AND (revenue_delta_gbp < 0 OR close_status = 'Lost')
ORDER BY revenue_delta_gbp ASC
LIMIT 25;

### KPI 6 — FY Recurring Revenue Retained
Sums GBP recurring revenue from existing customers (non-new logos) within the current fiscal year window.

In [ ]:
%%sql
WITH params AS (
    SELECT
        DATE '2025-01-01' AS fy_start,
        DATE '2026-01-01' AS fy_end
)
SELECT
    SUM(revenue_amount_gbp) AS retained_rr_gbp,
    COUNT(DISTINCT customer_id) AS retained_customers
FROM curated_customer_revenue c, params p
WHERE c.start_date >= p.fy_start
  AND c.start_date < p.fy_end
  AND COALESCE(is_new_logo, 0) = 0a
  AND c.close_status <> 'Lost';

### KPI 7 — Upgrade vs Loss Attribution (Current FY)
Breaks recurring revenue change into positive (upgrades/expansions) vs negative (contractions/churn) within the same fiscal window.

In [ ]:
%%sql
WITH params AS (
    SELECT
        DATE '2025-01-01' AS fy_start,
        DATE '2026-01-01' AS fy_end
)
SELECT
    SUM(CASE WHEN revenue_delta_gbp > 0 THEN revenue_delta_gbp ELSE 0 END) AS upgrade_rr_gbp,
    ABS(SUM(CASE WHEN revenue_delta_gbp < 0 THEN revenue_delta_gbp ELSE 0 END)) AS loss_rr_gbp,
    SUM(revenue_delta_gbp) AS net_growth_rr_gbp
FROM curated_customer_revenue c, params p
WHERE c.start_date >= p.fy_start
  AND c.start_date < p.fy_end;

### KPI 8 — Revenue YTD vs Prior Year
Compares GBP recurring revenue booked year-to-date against the same period in the prior fiscal year.

In [ ]:
%%sql
WITH params AS (
    SELECT
        DATE '2025-05-15' AS as_of_date
), cur AS (
    SELECT SUM(revenue_amount_gbp) AS ytd_rr_gbp
    FROM curated_customer_revenue c, params p
    WHERE c.start_date >= DATE_TRUNC('year', p.as_of_date)
      AND c.start_date < p.as_of_date
), prior AS (
    SELECT SUM(revenue_amount_gbp) AS prior_ytd_rr_gbp
    FROM curated_customer_revenue c, params p
    WHERE c.start_date >= DATE_TRUNC('year', p.as_of_date) - INTERVAL '1 year'
      AND c.start_date < p.as_of_date - INTERVAL '1 year'
 )
SELECT
    p.as_of_date,
    cur.ytd_rr_gbp,
    prior.prior_ytd_rr_gbp,
    cur.ytd_rr_gbp - prior.prior_ytd_rr_gbp AS ytd_delta_gbp
FROM params p, cur, prior;

### KPI 9 — Top Contributors by Customer and Product
Aggregates total recurring revenue by customer and product to highlight biggest drivers side-by-side.

In [ ]:
%%sql
WITH customer_rr AS (
    SELECT
        customer_id AS entity_id,
        COALESCE(customer_name, 'Unknown Customer') AS entity_name,
        SUM(revenue_amount_gbp) AS total_rr_gbp,
        'Customer' AS entity_type
    FROM curated_customer_revenue
    GROUP BY 1, 2
), product_rr AS (
    SELECT
        product_id AS entity_id,
        product_name AS entity_name,
        SUM(revenue_amount_gbp) AS total_rr_gbp,
        'Product' AS entity_type
    FROM curated_customer_revenue
    GROUP BY 1, 2
), combined AS (
    SELECT * FROM customer_rr
    UNION ALL
    SELECT * FROM product_rr
 )
SELECT
    entity_type,
    entity_id,
    entity_name,
    total_rr_gbp
FROM combined
ORDER BY entity_type, total_rr_gbp DESC
LIMIT 20;

### KPI 10 — Rolling 12-Month Recurring Revenue Trend
Builds a monthly time series and rolls it up over the prior 12 months to show directional momentum.

In [ ]:
%%sql
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', start_date) AS month_start,
        SUM(revenue_amount_gbp) AS monthly_rr_gbp
    FROM curated_customer_revenue
    GROUP BY 1
), rolling AS (
    SELECT
        month_start,
        monthly_rr_gbp,
        SUM(monthly_rr_gbp) OVER (
            ORDER BY month_start
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS rolling_12m_rr_gbp
    FROM monthly
 )
SELECT
    month_start,
    monthly_rr_gbp,
    rolling_12m_rr_gbp
FROM rolling
ORDER BY month_start;